In [2]:
import numpy as np
import pandas as pd

# Custom implementation

In [18]:
"""
Subgroup Discovery using Beam Search (Python)

This file contains a flexible implementation of subgroup discovery using beam search.
Features:
- Handles categorical and numeric attributes (numeric discretization via quantile thresholds)
- Beam search (width k) across conjunctions of attribute tests
- Supports binary or continuous target variable
- Configurable scoring functions: WRAcc (default for classification), precision, recall, F1, or custom callables
- Support for min_support and max_depth (max number of conditions per subgroup)
- Pretty printing + example usage on synthetic data

How to use:
- Import the SubgroupDiscovery class and call fit() with a pandas DataFrame and target column.

"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Tuple, Union
import pandas as pd
import numpy as np

Condition = Tuple[str, str, Any]  # (feature, operator, value)
Pattern = Tuple[Condition, ...]


@dataclass
class Subgroup:
    pattern: Pattern
    coverage: int
    target_stat: float  # mean target (continuous) or positives (discrete)
    score: float

    def __str__(self) -> str:
        conds = [f"{f} {op} {v}" for (f, op, v) in self.pattern]
        return f"IF {' AND '.join(conds)} -> coverage={self.coverage}, target_stat={self.target_stat:.4f}, score={self.score:.4f}"


class SubgroupDiscovery:
    def __init__(
        self,
        beam_width: int = 10,
        max_depth: int = 3,
        min_support: Union[int, float] = 0.05,
        numeric_thresholds: int = 5,
        scoring: Union[str, Callable[[int, float, int, float], float]] = "mean_diff_c",
        target_type: str = "continuous",  # "binary" or "continuous"
        target_positive: Any = 1,
        verbose: bool = False,
    ) -> None:
        """
        Parameters
        ----------
        beam_width: int
            Number of patterns to keep at each level of the beam.
        max_depth: int
            Maximum number of conditions in a pattern.
        min_support: int or float
            Minimum support for a pattern. If float between 0 and 1 -> fraction of records.
        numeric_thresholds: int
            Number of quantile-based thresholds to generate for numeric attributes.
        scoring: str or callable
            'wracc', 'precision', 'recall', 'f1' for binary target;
            or a custom callable with signature (n_cover, target_stat, n_total, global_stat) -> score
        target_type: str
            'binary' for classification, 'continuous' for regression-style subgroup discovery
        target_positive: value considered as positive class if binary
        verbose: bool
            Print progress messages.
        """
        self.beam_width = beam_width
        self.max_depth = max_depth
        self.min_support = min_support
        self.numeric_thresholds = numeric_thresholds
        self.verbose = verbose
        self.target_type = target_type
        self.target_positive = target_positive

        if isinstance(scoring, str):
            self.scoring_name = scoring
            if target_type == "binary":
                if scoring == "wracc":
                    self.scoring = self._score_wracc
                elif scoring == "precision":
                    self.scoring = self._score_precision
                elif scoring == "recall":
                    self.scoring = self._score_recall
                elif scoring == "f1":
                    self.scoring = self._score_f1
                else:
                    raise ValueError(f"Unknown scoring string for binary target: {scoring}")
            elif target_type == "continuous":
                if scoring == "mean_diff":
                    self.scoring = self._score_mean_diff
                    self.scoring_comparison = "global"
                elif scoring == "mean_diff_c":
                    self.scoring = self._score_mean_diff_c
                    self.scoring_comparison = "global"
                elif scoring == "tstat":
                    self.scoring = self._score_tstat
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_pos":
                    self.scoring = self._score_mean_diff_pos
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_neg":
                    self.scoring = self._score_mean_diff_neg
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_sqrt":
                    self.scoring = self._score_mean_diff_sqrt
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_pos_c":
                    self.scoring = self._score_mean_diff_pos_c
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_neg_c":
                    self.scoring = self._score_mean_diff_neg_c
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_pos_sqrt":
                    self.scoring = self._score_mean_diff_pos_sqrt
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_neg_sqrt":
                    self.scoring = self._score_mean_diff_neg_sqrt
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_klcsw":
                    self.scoring = self._score_mean_diff_klcsw
                    self.scoring_comparison = "complement"
                elif scoring == "mean_diff_bkw":
                    self.scoring = self._score_mean_diff_bkw
                    self.scoring_comparison = "complement"
                else:
                    raise ValueError(f"Unknown scoring string for continuous target: {scoring}")
            else:
                raise ValueError("target_type must be 'binary' or 'continuous'")
        else:
            self.scoring_name = "custom"
            self.scoring_comparison = "unspecified"
            self.scoring = scoring

        # internals
        self.feature_types: Dict[str, str] = {}
        self.thresholds: Dict[str, List[float]] = {}
        self.best_subgroups: List[Subgroup] = []

    # ----------------------- scoring functions -----------------------
    def _score_wracc(self, n_cover: int, n_pos: int, n_total: int, n_pos_total: int) -> float:
        if n_cover == 0:
            return 0.0
        p_global = n_pos_total / n_total
        p_sub = n_pos / n_cover
        coverage_frac = n_cover / n_total
        return coverage_frac * (p_sub - p_global)

    def _score_precision(self, n_cover: int, n_pos: int, n_total: int, n_pos_total: int) -> float:
        if n_cover == 0:
            return 0.0
        return n_pos / n_cover

    def _score_recall(self, n_cover: int, n_pos: int, n_total: int, n_pos_total: int) -> float:
        if n_pos_total == 0:
            return 0.0
        return n_pos / n_pos_total

    def _score_f1(self, n_cover: int, n_pos: int, n_total: int, n_pos_total: int) -> float:
        prec = self._score_precision(n_cover, n_pos, n_total, n_pos_total)
        rec = self._score_recall(n_cover, n_pos, n_total, n_pos_total)
        if prec + rec == 0:
            return 0.0
        return 2 * (prec * rec) / (prec + rec)

    def _score_mean_diff(self, n_cover: int, mean_sub: float, n_total: int, mean_global: float) -> float:
        if n_cover == 0:
            return 0.0
        return abs(mean_sub - mean_global)
    
    def _score_mean_diff_c(self, n_cover: int, mean_sub: float, n_total: int, mean_global: float) -> float:
        if n_cover == 0:
            return 0.0
        # difference in means, scaled by coverage fraction
        coverage_frac = n_cover / n_total
        return coverage_frac * abs(mean_sub - mean_global)

    def _score_tstat(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        variance_subgroup: float = target_subgroup.var()
        variance_complement: float = target_complement.var()
        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size

        if (n_subgroup == 0 or n_complement == 0):
            return 0

        SE: float = np.sqrt(variance_subgroup/n_subgroup + variance_complement/n_complement)
        t: float = (mean_subgroup - mean_complement) / SE
        
        return np.abs(t)

    def _score_mean_diff_pos(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:

        if target_subgroup.size == 0 or target_complement.size == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()

        return mean_subgroup - mean_complement    

    def _score_mean_diff_neg(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:

        if target_subgroup.size == 0 or target_complement.size == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()

        return mean_complement - mean_subgroup

    def _score_mean_diff_sqrt(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:
        
        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        return np.sqrt(cov) * np.abs(mean_subgroup - mean_complement)

    def _score_mean_diff_pos_c(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:

        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        return cov * (mean_subgroup - mean_complement)    

    def _score_mean_diff_neg_c(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:

        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        return cov * (mean_complement - mean_subgroup)   

    def _score_mean_diff_pos_sqrt(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:
        
        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        return np.sqrt(cov) * (mean_subgroup - mean_complement)

    def _score_mean_diff_neg_sqrt(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:
        
        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        return np.sqrt(cov) * (mean_complement - mean_subgroup)

    def _score_mean_diff_klcsw(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:
        
        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        # Peak at 30% coverage
        center: float = 0.30

        # Suitable sharpness
        gamma: int = 12
        
        D: float = cov * np.log(cov/center) + (1 - cov) * np.log((1 - cov)/(1 - center))
        klcsw: float = np.e**(-gamma * D)

        return klcsw * np.abs(mean_complement - mean_subgroup)

    def _score_mean_diff_bkw(self, target_subgroup: pandas.DataFrame, target_complement: pandas.DataFrame) -> float:

        n_subgroup: int = target_subgroup.size
        n_complement: int = target_complement.size
        n_total: int = n_subgroup + n_complement
        
        if  n_subgroup == 0 or n_complement == 0:
            return 0
        
        mean_subgroup: float = target_subgroup.mean()
        mean_complement: float = target_complement.mean()
        cov: float = n_subgroup / n_total

        # Chosen for a peak around 23% and a suitable sharpness
        alpha: float = 0.7
        beta: float = 2.3

        num: float = cov**alpha * (1 - cov)**beta
        denom: float = ((alpha / (alpha + beta))** alpha) * ((beta / (alpha + beta))**beta)

        bkw: float = num/denom

        return bkw * np.abs(mean_subgroup - mean_complement)
        
    # ----------------------- helpers -----------------------
    def _detect_feature_types(self, df: pd.DataFrame, exclude: str) -> None:
        for col in df.columns:
            if col == exclude:
                continue
            if pd.api.types.is_numeric_dtype(df[col]):
                self.feature_types[col] = "numeric"
            else:
                self.feature_types[col] = "categorical"

    def _generate_thresholds(self, df: pd.DataFrame) -> None:
        self.thresholds = {}
        for f, t in self.feature_types.items():
            if t == "numeric":
                vals = df[f].dropna().unique()
                if len(vals) <= self.numeric_thresholds:
                    thr = sorted(vals)
                else:
                    # quantile-based thresholds
                    qs = np.linspace(0, 1, self.numeric_thresholds + 2)[1:-1]
                    thr = sorted(pd.Series(df[f].dropna()).quantile(qs).unique())
                self.thresholds[f] = thr
            else:
                self.thresholds[f] = sorted(df[f].dropna().unique())

    def _apply_condition(self, df: pd.DataFrame, cond: Condition) -> pd.Series:
        f, op, v = cond
        if op == "==":
            return df[f] == v
        elif op == "<=":
            return df[f] <= v
        elif op == ">":
            return df[f] > v
        else:
            raise ValueError(f"Unknown operator: {op}")

    def _apply_pattern_mask(self, df: pd.DataFrame, pattern: Pattern) -> np.ndarray:
        if len(pattern) == 0:
            return np.ones(len(df), dtype=bool)
        masks = [self._apply_condition(df, cond) for cond in pattern]
        return np.logical_and.reduce(masks)

    def _pattern_to_str(self, pattern: Pattern) -> str:
        return " AND ".join([f"{f}{op}{v}" for (f, op, v) in pattern])

    # ----------------------- beam search core -----------------------
    def fit(self, df: pd.DataFrame, target: str, top_k: int = 10) -> List[Subgroup]:
        """
        Run beam search to find top_k subgroups.

        Returns list of Subgroup objects sorted by score descending.
        """
        df = df.copy()
        n_total = len(df)
        if n_total == 0:
            return []

        # interpret min_support
        if 0 < self.min_support < 1:
            min_sup_count = max(1, int(np.ceil(self.min_support * n_total)))
        else:
            min_sup_count = int(self.min_support)

        # globals
        self._detect_feature_types(df, exclude=target)
        self._generate_thresholds(df)

        if self.target_type == "binary":
            global_stat = int((df[target] == self.target_positive).sum())
        else:  # continuous
            global_stat = df[target].mean()

        if self.verbose:
            print(f"n_total={n_total}, global_stat={global_stat}, min_sup_count={min_sup_count}")

        # initial candidates: single conditions
        initial_candidates: List[Pattern] = []
        for f, t in self.feature_types.items():
            for val in self.thresholds[f]:
                if t == "numeric":
                    initial_candidates.append(((f, "<=", float(val)),))
                    initial_candidates.append(((f, ">", float(val)),))
                else:
                    initial_candidates.append(((f, "==", val),))

        # evaluate initial candidates and form first beam
        scored: List[Subgroup] = []
        for p in initial_candidates:
            mask = self._apply_pattern_mask(df, p)
            n_cover = int(mask.sum())
            if n_cover < min_sup_count:
                continue
            if self.target_type == "binary":
                n_pos = int(((df.loc[mask, target]) == self.target_positive).sum())
                stat = n_pos
                score = self.scoring(n_cover, n_pos, n_total, global_stat)
            else:
                mean_sub = df.loc[mask, target].mean()
                stat = mean_sub
                target_sub = df.loc[mask, target]
                target_complement = df.iloc[np.delete(df.index.values, df.loc[mask, target].index.values)][target]
                if self.scoring_comparison == "global":
                    score = self.scoring(n_cover, mean_sub, n_total, global_stat)
                elif self.scoring_comparison == "complement":
                    score = self.scoring(target_sub, target_complement)
            scored.append(Subgroup(p, n_cover, stat, score))

        # sort beam by score
        scored.sort(key=lambda s: s.score, reverse=True)
        beam = [s.pattern for s in scored[: self.beam_width]]
        self.best_subgroups = scored[:top_k]

        
        # iterative expansion
        for depth in range(2, self.max_depth + 1):
            if len(beam) == 0:
                break

            candidates: List[Pattern] = []
            seen_pattern_keys: set = set()

            for pattern in beam:
                used_features = {c[0] for c in pattern}
                for f, t in self.feature_types.items():
                    if f in used_features:
                        continue
                    ops = ("<=", ">") if t == "numeric" else ("==",)
                    for val in self.thresholds[f]:
                        for op in ops:
                            v = float(val) if t == "numeric" else val
                            new_pat = tuple(list(pattern) + [(f, op, v)])
                            
                            new_pat_sorted = tuple(sorted(new_pat, key=lambda x: (x[0], x[1], str(x[2]))))
            
                            key = frozenset(new_pat_sorted)
            
                            if key in seen_pattern_keys:
                                continue
            
                            seen_pattern_keys.add(key)
                            candidates.append(new_pat_sorted)

            # evaluate candidates and pick top beam_width
            scored_next: List[Subgroup] = []
            for p in candidates:
                mask = self._apply_pattern_mask(df, p)
                n_cover = int(mask.sum())
                if n_cover < min_sup_count:
                    continue
                if self.target_type == "binary":
                    n_pos = int(((df.loc[mask, target]) == self.target_positive).sum())
                    stat = n_pos
                    score = self.scoring(n_cover, n_pos, n_total, global_stat)
                else:
                    mean_sub = df.loc[mask, target].mean()
                    stat = mean_sub
                    target_sub = df.loc[mask, target]
                    target_complement = df.iloc[np.delete(df.index.values, df.loc[mask, target].index.values)][target]
                    if self.scoring_comparison == "global":
                        score = self.scoring(n_cover, mean_sub, n_total, global_stat)
                    elif self.scoring_comparison == "complement":
                        score = self.scoring(target_sub, target_complement)
            
                scored_next.append(Subgroup(p, n_cover, stat, score))

            scored_next.sort(key=lambda s: s.score, reverse=True)
            beam = [s.pattern for s in scored_next[: self.beam_width]]

            combined = self.best_subgroups + scored_next
            combined.sort(key=lambda s: s.score, reverse=True)
            self.best_subgroups = combined[:top_k]

        self.best_subgroups.sort(key=lambda s: s.score, reverse=True)
        return self.best_subgroups

# Application on our data

In [4]:
df = pd.read_csv("feature_engineered_data/df_final.csv")
df.head()

,season,episode,title,episode_script,episode_script_clean,avg_words_per_line,vocab_richness,character_lines_MONICA,character_lines_RACHEL,character_lines_ROSS,...,Central Perk,City spaces,Hallways,Hospitals,Jobs,Main apartments,Other lodging,Social Life,Transport,stars
0,1,1,The One Where Monica Gets a Roommate: The Pilot,"[SCENE NOTE: Scene: Central Perk, Chandler, Jo...","[SCENE NOTE: Scene: Central Perk, Chandler, Jo...",11.026667,0.394800,0.243333,0.163333,0.163333,...,0.250000,0.000000,0.0,0.000000,0.083333,0.500000,0.0,0.083333,0.083333,8.3
1,1,2,The One with the Sonogram at the End,"[SCENE NOTE: Scene Central Perk, everyone's th...","[SCENE NOTE: Scene Central Perk, everyone's th...",9.572016,0.448839,0.115226,0.156379,0.259259,...,0.181818,0.000000,0.0,0.272727,0.272727,0.272727,0.0,0.000000,0.000000,8.1
2,1,3,The One with the Thumb,"[SCENE NOTE: Scene: Central Perk, everyone but...","[SCENE NOTE: Scene: Central Perk, everyone but...",9.104651,0.442316,0.201550,0.100775,0.127907,...,0.333333,0.133333,0.0,0.000000,0.200000,0.266667,0.0,0.066667,0.000000,8.2
3,1,4,The One with George Stephanopoulos,"[SCENE NOTE: Scene: Central Perk, everyone is ...","[SCENE NOTE: Scene: Central Perk, everyone is ...",9.464286,0.459958,0.190476,0.170635,0.158730,...,0.187500,0.187500,0.0,0.250000,0.000000,0.375000,0.0,0.000000,0.000000,8.1
4,1,5,The One with the East German Laundry Detergent,"[SCENE NOTE: Scene: Central Perk, all six are ...","[SCENE NOTE: Scene: Central Perk, all six are ...",10.268908,0.392799,0.138655,0.155462,0.168067,...,0.375000,0.187500,0.0,0.000000,0.000000,0.125000,0.0,0.312500,0.000000,8.5


In [5]:
df.columns

Index(['season', 'episode', 'title', 'episode_script', 'episode_script_clean',
       'avg_words_per_line', 'vocab_richness', 'character_lines_MONICA',
       'character_lines_RACHEL', 'character_lines_ROSS',
       'character_lines_CHANDLER', 'character_lines_JOEY',
       'character_lines_PHOEBE', 'character_lines_OTHER', 'emotion_joy',
       'emotion_sadness', 'emotion_anger', 'emotion_fear', 'emotion_surprise',
       'emotion_disgust', 'emotion_neutral', 'sentiment_neg', 'sentiment_neu',
       'sentiment_pos', 'sentiment_compound', 'sentiment_trajectory_slope',
       'humor', 'exchanges_JOEY_CHANDLER', 'exchanges_RACHEL_MONICA',
       'exchanges_ROSS_RACHEL', 'exchanges_MONICA_CHANDLER',
       'exchanges_PHOEBE_JOEY', 'Central Perk', 'City spaces', 'Hallways',
       'Hospitals', 'Jobs', 'Main apartments', 'Other lodging', 'Social Life',
       'Transport', 'stars'],
      dtype='object')

In [6]:
subgroup_descriptors = ["avg_words_per_line", "vocab_richness", 
                                 "character_lines_MONICA", 'character_lines_RACHEL', 'character_lines_ROSS', 'character_lines_CHANDLER', 
                                 'character_lines_JOEY', 'character_lines_PHOEBE', 'character_lines_OTHER',
                                 'emotion_joy', 'emotion_sadness', 'emotion_anger', 'emotion_fear', 'emotion_surprise', 'emotion_disgust', 'emotion_neutral',
                                 'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'sentiment_trajectory_slope',
                                 'humor',
                                 'exchanges_JOEY_CHANDLER', 'exchanges_RACHEL_MONICA', 'exchanges_ROSS_RACHEL', 'exchanges_MONICA_CHANDLER', 'exchanges_PHOEBE_JOEY',
                                 'Central Perk', 'City spaces', 'Hallways', 'Hospitals', 'Jobs', 'Main apartments', 'Other lodging', 'Social Life', 'Transport']

descriptors_and_target = subgroup_descriptors
descriptors_and_target.append("stars")

## Scored by mean difference scaled by coverage

In [6]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_c", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [7]:
for s in top:
    print(s)
    print("\n")

IF Central Perk > 0.05366357069143439 AND emotion_anger <= 0.07928996565752607 AND emotion_disgust > 0.03558859298178982 -> coverage=106, target_stat=8.2962, score=0.0731


IF Central Perk > 0.1176470588235294 AND emotion_anger <= 0.09343565798345005 AND exchanges_PHOEBE_JOEY <= 16.0 -> coverage=106, target_stat=8.3009, score=0.0709


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 AND avg_words_per_line > 9.118298996014051 -> coverage=141, target_stat=8.3404, score=0.0708


IF Central Perk > 0.05366357069143439 AND emotion_anger <= 0.07928996565752607 AND exchanges_JOEY_CHANDLER <= 31.66666666666663 -> coverage=113, target_stat=8.3124, score=0.0702


IF Central Perk > 0.05366357069143439 AND avg_words_per_line > 9.118298996014051 AND emotion_anger <= 0.09343565798345005 -> coverage=138, target_stat=8.3406, score=0.0692


IF Central Perk <= 0.1666666666666666 AND character_lines_JOEY <= 0.17635569852941174 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=88, tar

Note that these subgroups are all quite big in coverage because of the scaling of the quality with coverage. We might find more interesting subgroups without the coverage scaling.

In [8]:
# Since Central Perk shows up in every top subgroup,  let's take a closer look
print(f"Median stars for Central Perk > 0.053: {df[df["Central Perk"] > 0.053]["stars"].median()}")
print(f"Median stars for Central Perk <= 0.053: {df[df["Central Perk"] <= 0.053]["stars"].median()}")

print("=====================================================================")

# Same for Hallways
print(f"Median stars for Hallways <= 0.077: {df[df["Hallways"] <= 0.077]["stars"].median()}")
print(f'Median stars for Hallways > 0.077: {df[df["Hallways"] > 0.077]["stars"].median()}')

print("=====================================================================")

# Same for exchanges between Joey and Chandler
print(f"Median stars for exchanges_JOEY_CHANDLER <= 31: {df[df["exchanges_JOEY_CHANDLER"] <= 31]["stars"].median()}")
print(f"Median stars for exchanges_JOEY_CHANDLER > 31: {df[df["exchanges_JOEY_CHANDLER"] > 31]["stars"].median()}")

Median stars for Central Perk > 0.053: 8.3
Median stars for Central Perk <= 0.053: 8.7
Median stars for Hallways <= 0.077: 8.4
Median stars for Hallways > 0.077: 8.6
Median stars for exchanges_JOEY_CHANDLER <= 31: 8.4
Median stars for exchanges_JOEY_CHANDLER > 31: 8.5


The differences for hallways and joey_chandler exchanges don't seem so big. They are likely more interesting in the interaction with the other descriptors.  

## Scored by mean difference without coverage

In [9]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [10]:
for s in top:
    print(s)
    print("\n")

IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 9.118298996014051 AND character_lines_CHANDLER > 0.12053264834684568 -> coverage=10, target_stat=9.1000, score=0.6411


IF Central Perk <= 0.1176470588235294 AND Social Life <= 0.0 AND avg_words_per_line <= 9.118298996014051 -> coverage=11, target_stat=9.0818, score=0.6229


IF Central Perk <= 0.1176470588235294 AND Social Life <= 0.0714285714285714 AND avg_words_per_line <= 9.118298996014051 -> coverage=11, target_stat=9.0818, score=0.6229


IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 9.118298996014051 AND character_lines_OTHER <= 0.15568986413216815 -> coverage=13, target_stat=9.0769, score=0.6180


IF Central Perk <= 0.1176470588235294 AND Hospitals <= 0.0 AND character_lines_ROSS > 0.19742720946069386 -> coverage=10, target_stat=9.0500, score=0.5911


IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 9.118298996014051 AND humor <= 0.634012618435226 -> coverage=11, target_stat=9.0455, s

Since I set the minimum coverage to 1/20th and pretty much all of the top 15 subgroups have exactly (rounded) 1/20th coverage, I think we will get wildly different subgroups per mimimum coverage we choose.

## Scored by welch's t

Why welch's t? It accounts for size and variance of the subgroup by nature.

In [7]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="tstat", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [8]:
for s in top:
    print(s)
    print("\n")

IF exchanges_JOEY_CHANDLER <= 18.66666666666663 AND exchanges_MONICA_CHANDLER <= 4.0 AND exchanges_ROSS_RACHEL > 6.0 -> coverage=11, target_stat=8.1455, score=9.7429


IF Central Perk > 0.2307692307692307 AND emotion_sadness > 0.05581657169458569 AND exchanges_ROSS_RACHEL <= 24.0 -> coverage=10, target_stat=8.1400, score=9.6412


IF character_lines_MONICA > 0.15588067444876783 AND character_lines_ROSS <= 0.16798958333333333 AND exchanges_MONICA_CHANDLER <= 4.0 -> coverage=10, target_stat=8.0700, score=9.5322


IF character_lines_JOEY <= 0.14921023965141605 AND exchanges_MONICA_CHANDLER <= 4.0 AND exchanges_PHOEBE_JOEY > 3.0 -> coverage=14, target_stat=8.1500, score=9.3124


IF avg_words_per_line > 9.118298996014051 AND character_lines_MONICA > 0.15588067444876783 AND exchanges_MONICA_CHANDLER <= 4.0 -> coverage=11, target_stat=8.0818, score=8.9872


IF Central Perk > 0.27693602693602687 AND Jobs > 0.08119658119658092 AND character_lines_JOEY <= 0.14921023965141605 -> coverage=10, targe

## Scored by negative mean difference

In [13]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_neg", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [14]:
for s in top:
    print(s)
    print("\n")

IF avg_words_per_line > 9.118298996014051 AND character_lines_PHOEBE <= 0.10295154334942555 AND vocab_richness > 0.40793454382631256 -> coverage=10, target_stat=7.9100, score=0.5732


IF character_lines_PHOEBE <= 0.10295154334942555 AND humor > 0.634012618435226 AND vocab_richness > 0.40793454382631256 -> coverage=11, target_stat=7.9273, score=0.5576


IF Jobs > 0.0 AND emotion_sadness > 0.03666910494504636 AND humor > 0.6980063008416796 -> coverage=10, target_stat=7.9300, score=0.5523


IF Hospitals <= 0.0 AND exchanges_JOEY_CHANDLER > 18.66666666666663 AND vocab_richness > 0.40793454382631256 -> coverage=10, target_stat=7.9300, score=0.5523


IF Main apartments > 0.4166666666666667 AND sentiment_pos > 0.1563726869161651 AND vocab_richness > 0.40793454382631256 -> coverage=10, target_stat=7.9300, score=0.5523


IF emotion_surprise > 0.1835716614841136 AND sentiment_neg <= 0.0452325188258798 AND vocab_richness > 0.3974609859400576 -> coverage=11, target_stat=7.9545, score=0.5290


IF C

## Scored by positive mean difference

In [15]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_pos", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [16]:
for s in top:
    print(s)
    print("\n")

IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 9.118298996014051 AND character_lines_CHANDLER > 0.12053264834684568 -> coverage=10, target_stat=9.1000, score=0.6695


IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 9.118298996014051 AND character_lines_OTHER <= 0.15568986413216815 -> coverage=13, target_stat=9.0769, score=0.6541


IF Central Perk <= 0.1176470588235294 AND Social Life <= 0.0 AND avg_words_per_line <= 9.118298996014051 -> coverage=11, target_stat=9.0818, score=0.6534


IF Central Perk <= 0.1176470588235294 AND Social Life <= 0.0714285714285714 AND avg_words_per_line <= 9.118298996014051 -> coverage=11, target_stat=9.0818, score=0.6534


IF Central Perk <= 0.1176470588235294 AND Hospitals <= 0.0 AND character_lines_ROSS > 0.19742720946069386 -> coverage=10, target_stat=9.0500, score=0.6173


IF Central Perk <= 0.1176470588235294 AND Main apartments > 0.4166666666666667 AND avg_words_per_line <= 9.118298996014051 -> coverage=11, target_stat

## Scored by coverage weighted negative mean difference

In [17]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_neg_c", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [18]:
for s in top:
    print(s)
    print("\n")

IF Central Perk > 0.05366357069143439 -> coverage=196, target_stat=8.3974, score=0.3011


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 -> coverage=178, target_stat=8.3876, score=0.2187


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 -> coverage=166, target_stat=8.3693, score=0.2125


IF Central Perk > 0.05366357069143439 AND avg_words_per_line > 9.118298996014051 -> coverage=167, target_stat=8.3713, score=0.2121


IF Central Perk > 0.05366357069143439 AND emotion_disgust > 0.03558859298178982 -> coverage=163, target_stat=8.3687, score=0.2014


IF Central Perk > 0.05366357069143439 AND emotion_anger <= 0.09343565798345005 -> coverage=164, target_stat=8.3726, score=0.1967


IF Central Perk > 0.05366357069143439 AND Other lodging <= 0.1666666666666666 -> coverage=166, target_stat=8.3807, score=0.1854


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 AND emotion_anger <= 0.09343565798345005 -> cov

## Scored by coverage weighted positive mean difference

In [19]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_pos_c", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [20]:
for s in top:
    print(s)
    print("\n")

IF exchanges_MONICA_CHANDLER > 4.0 -> coverage=187, target_stat=8.5064, score=0.1813


IF Central Perk <= 0.27693602693602687 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=155, target_stat=8.5419, score=0.1589


IF Central Perk <= 0.27693602693602687 -> coverage=196, target_stat=8.4903, score=0.1539


IF Jobs <= 0.2307692307692307 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=164, target_stat=8.5262, score=0.1533


IF character_lines_ROSS > 0.10338481162589404 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=154, target_stat=8.5357, score=0.1443


IF exchanges_MONICA_CHANDLER > 4.0 AND humor <= 0.6980063008416796 -> coverage=156, target_stat=8.5321, score=0.1426


IF Central Perk <= 0.27693602693602687 AND Jobs <= 0.2307692307692307 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=134, target_stat=8.5672, score=0.1422


IF Central Perk <= 0.27693602693602687 AND character_lines_RACHEL > 0.1062000929257566 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=130, target_stat=8.5746, score=

## Scored by square root coverage weighted mean difference

In [21]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_sqrt", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [22]:
for s in top:
    print(s)
    print("\n")

IF Central Perk > 0.05366357069143439 -> coverage=196, target_stat=8.3974, score=0.3304


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 -> coverage=166, target_stat=8.3693, score=0.2534


IF Central Perk > 0.05366357069143439 AND avg_words_per_line > 9.118298996014051 -> coverage=167, target_stat=8.3713, score=0.2522


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 -> coverage=178, target_stat=8.3876, score=0.2518


IF Central Perk > 0.05366357069143439 AND emotion_disgust > 0.03558859298178982 -> coverage=163, target_stat=8.3687, score=0.2423


IF Central Perk > 0.05366357069143439 AND emotion_anger <= 0.09343565798345005 -> coverage=164, target_stat=8.3726, score=0.2359


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 AND avg_words_per_line > 9.118298996014051 -> coverage=141, target_stat=8.3404, score=0.2275


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 AND emoti

## Scored by square root coverage weighted positive mean difference

In [23]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_pos_sqrt", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [24]:
for s in top:
    print(s)
    print("\n")

IF exchanges_MONICA_CHANDLER > 4.0 -> coverage=187, target_stat=8.5064, score=0.2037


IF Central Perk <= 0.27693602693602687 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=155, target_stat=8.5419, score=0.1961


IF Central Perk <= 0.27693602693602687 AND character_lines_RACHEL > 0.1062000929257566 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=130, target_stat=8.5746, score=0.1912


IF Central Perk <= 0.27693602693602687 AND Jobs <= 0.2307692307692307 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=134, target_stat=8.5672, score=0.1888


IF Central Perk <= 0.27693602693602687 AND exchanges_MONICA_CHANDLER > 4.0 AND humor <= 0.6980063008416796 -> coverage=130, target_stat=8.5723, score=0.1874


IF Jobs <= 0.2307692307692307 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=164, target_stat=8.5262, score=0.1839


IF Central Perk <= 0.27693602693602687 AND emotion_disgust <= 0.053389852674700124 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=102, target_stat=8.6167, score=0.1827


IF Ce

## Scored by square root coverage weighted negative mean difference

In [25]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_neg_sqrt", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [26]:
for s in top:
    print(s)
    print("\n")

IF Central Perk > 0.05366357069143439 -> coverage=196, target_stat=8.3974, score=0.3304


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 -> coverage=166, target_stat=8.3693, score=0.2534


IF Central Perk > 0.05366357069143439 AND avg_words_per_line > 9.118298996014051 -> coverage=167, target_stat=8.3713, score=0.2522


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 -> coverage=178, target_stat=8.3876, score=0.2518


IF Central Perk > 0.05366357069143439 AND emotion_disgust > 0.03558859298178982 -> coverage=163, target_stat=8.3687, score=0.2423


IF Central Perk > 0.05366357069143439 AND emotion_anger <= 0.09343565798345005 -> coverage=164, target_stat=8.3726, score=0.2359


IF Central Perk > 0.05366357069143439 AND Hallways <= 0.0769230769230769 AND avg_words_per_line > 9.118298996014051 -> coverage=141, target_stat=8.3404, score=0.2275


IF Central Perk > 0.05366357069143439 AND Main apartments <= 0.6666666666666666 AND emoti

## Scored by Kullback-Leibler centered support weighted mean difference

In [13]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_klcsw", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [14]:
for s in top:
    print(s)
    print("\n")

IF Central Perk <= 0.1666666666666666 AND emotion_disgust <= 0.053389852674700124 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=68, target_stat=8.6912, score=0.3250


IF Central Perk <= 0.1666666666666666 AND emotion_disgust <= 0.053389852674700124 AND exchanges_RACHEL_MONICA > 2.0 -> coverage=63, target_stat=8.7048, score=0.3249


IF Central Perk <= 0.1176470588235294 AND emotion_anger > 0.04208676309499123 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=62, target_stat=8.7032, score=0.3182


IF Central Perk <= 0.1176470588235294 AND character_lines_ROSS > 0.10338481162589404 AND emotion_disgust <= 0.06069374284053404 -> coverage=56, target_stat=8.7304, score=0.3164


IF Central Perk <= 0.1176470588235294 AND character_lines_RACHEL > 0.1062000929257566 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=56, target_stat=8.7286, score=0.3144


IF Central Perk <= 0.1176470588235294 AND character_lines_ROSS > 0.10338481162589404 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=54, target_sta

## Scored by Beta Kernel Weighted mean difference

In [19]:
sd = SubgroupDiscovery(beam_width=500, max_depth=3, min_support=10, numeric_thresholds=5, scoring="mean_diff_bkw", target_type="continuous", verbose=True)
top = sd.fit(df[descriptors_and_target], target="stars", top_k=15)

n_total=236, global_stat=8.458898305084746, min_sup_count=10


In [20]:
for s in top:
    print(s)
    print("\n")

IF Central Perk <= 0.05366357069143439 AND emotion_joy > 0.11352088232925764 AND exchanges_MONICA_CHANDLER > 4.0 -> coverage=32, target_stat=8.8500, score=0.4078


IF Central Perk <= 0.1176470588235294 AND Jobs <= 0.0 AND exchanges_ROSS_RACHEL > 12.0 -> coverage=31, target_stat=8.8548, score=0.4063


IF Central Perk <= 0.1176470588235294 AND Jobs <= 0.0 AND emotion_joy > 0.11352088232925764 -> coverage=44, target_stat=8.7932, score=0.4026


IF Central Perk <= 0.1176470588235294 AND Jobs <= 0.08119658119658092 AND exchanges_ROSS_RACHEL > 12.0 -> coverage=34, target_stat=8.8324, score=0.4011


IF Central Perk <= 0.1176470588235294 AND avg_words_per_line <= 10.227081021087681 AND exchanges_ROSS_RACHEL > 12.0 -> coverage=29, target_stat=8.8621, score=0.3999


IF Central Perk <= 0.1176470588235294 AND character_lines_OTHER <= 0.15568986413216815 AND exchanges_ROSS_RACHEL > 12.0 -> coverage=29, target_stat=8.8621, score=0.3999


IF Central Perk <= 0.1176470588235294 AND emotion_surprise > 0.

Beta Kernel Weight in purple (alpha = 0.7, beta = 2.3), Kullback-Leibler centered support weight in red (gamma = 12, center = 0.3).
![bkw_vs_klcsw](bkw_vs_klcsw.png)